In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import re
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report
from imblearn.over_sampling import SMOTE
import joblib
from time import time
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import StackingClassifier

# locate dataset
project_root = Path.cwd()
data_path = None
for parent in [project_root] + list(project_root.parents):
    candidate = parent / 'Data' / 'Numerical' / 'Data' / '65_Nutrients_Data.csv'
    if candidate.exists():
        data_path = candidate
        project_root = parent
        break

if data_path is None:
    raise FileNotFoundError('Could not find 65_Nutrients_Data.csv under any parent of '+str(Path.cwd()))

print('Using data file:', data_path)
df = pd.read_csv(data_path)
# sanitize columns for LightGBM compatibility
df.columns = [re.sub(r'[^0-9a-zA-Z_]', '_', str(c)) for c in df.columns]
label_col = 'novaclass'
if label_col not in df.columns:
    raise KeyError(f"{label_col} column missing after sanitizing columns: {df.columns.tolist()}")

X = df.drop(columns=[label_col])
y = df[label_col].astype(int)
min_label = int(y.min())
if min_label != 0:
    y = y - min_label

# split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print('train/test sizes', X_train.shape, X_test.shape)

# resample with SMOTE
sm = SMOTE(random_state=42)
X_res, y_res = sm.fit_resample(X_train, y_train)
print('Resampled train sizes', X_res.shape, np.bincount(y_res))

# prepare base estimators (robust imports)
estimators = []
# LightGBM
try:
    from lightgbm import LGBMClassifier
    estimators.append(('lgbm', LGBMClassifier(random_state=42)))
except Exception as e:
    print('LightGBM not available:', e)
# XGBoost
try:
    from xgboost import XGBClassifier
    estimators.append(('xgb', XGBClassifier(use_label_encoder=False, eval_metric='mlogloss', random_state=42)))
except Exception as e:
    print('XGBoost not available:', e)
# CatBoost
try:
    from catboost import CatBoostClassifier
    estimators.append(('cat', CatBoostClassifier(random_seed=42, verbose=0)))
except Exception as e:
    print('CatBoost not available:', e)
# fallback
if len(estimators) == 0:
    try:
        from sklearn.ensemble import HistGradientBoostingClassifier
        estimators.append(('hgb', HistGradientBoostingClassifier(random_state=42)))
        print('Using fallback HistGradientBoosting as base estimator')
    except Exception as e2:
        raise ImportError('No base estimators available (lightgbm/xgboost/catboost/hgb)')

# stacking with passthrough to give meta-learner access to original features
stack = StackingClassifier(estimators=estimators, final_estimator=LogisticRegression(max_iter=2000), n_jobs=-1, passthrough=True)

print('Training extended stacking ensemble with:', [name for name,_ in estimators])
t0 = time()
stack.fit(X_res, y_res)
t1 = time()

y_pred = stack.predict(X_test)
print('\nClassification report (extended stack):')
print(classification_report(y_test, y_pred))
print('Test f1_macro (extended stack):', f1_score(y_test, y_pred, average='macro'))
print('Train time (s):', round(t1-t0,2))

out_dir_stack2 = project_root / 'models' / 'numerical_65_stack_passthrough' / 'smote'
out_dir_stack2.mkdir(parents=True, exist_ok=True)
joblib.dump(stack, out_dir_stack2 / 'stack_passthrough_65_numerical_smote.joblib')
print('Saved extended stacking model to', out_dir_stack2)

Using data file: c:\Users\Divyeh\OneDrive\Desktop\ML_IP\NOVA_Food_Processing\Data\Numerical\Data\65_Nutrients_Data.csv
train/test sizes (2376, 65) (594, 65)
Resampled train sizes (6760, 65) [1690 1690 1690 1690]
Training extended stacking ensemble with: ['lgbm', 'xgb', 'cat']

Classification report (extended stack):
              precision    recall  f1-score   support

           0       0.82      0.90      0.86        68
           1       0.62      0.91      0.74        11
           2       0.82      0.75      0.79        93
           3       0.96      0.95      0.96       422

    accuracy                           0.91       594
   macro avg       0.81      0.88      0.84       594
weighted avg       0.92      0.91      0.91       594

Test f1_macro (extended stack): 0.8356043201233396
Train time (s): 323.54
Saved extended stacking model to c:\Users\Divyeh\OneDrive\Desktop\ML_IP\NOVA_Food_Processing\models\numerical_65_stack_passthrough\smote


C:\Users\Divyeh\AppData\Roaming\Python\Python313\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 2000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=2000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
